In [78]:
import pandas as pd
import os

# Caminhos de entrada e saída
json_path = "dados_cotacao.json"
output_dir = "silver-json-to_parquet/cotacao_parquet"

# Criar pasta de saída se não existir
os.makedirs(output_dir, exist_ok=True)
print("Lendo Json")

df = pd.read_json(json_path)

df['data_cotacao'] = pd.to_datetime(df['data_cotacao'], format="%d/%m/%Y", errors='coerce')
df['ano'] = df['data_cotacao'].dt.year
df["ano"] = df["ano"].astype("int32")
df["mes"] = df['data_cotacao'].dt.month
df["valor"] = df["valor"].replace(".","").replace(",",".")
df = df.dropna(subset=['ano', 'valor'])

total_parquet = 0
anos_processados = []

print("Gerando arquivos Parquet por ano:")
for ano, grupo in df.groupby('ano'):

        path_ano = os.path.join(output_dir, f"ano={ano}")
        os.makedirs(path_ano, exist_ok=True)

        parquet_path = os.path.join(path_ano, f"dados_cotacao_{ano}.parquet")
        grupo = grupo.astype({
            "ano": "int32",
            "mes": "int16",
            "valor": "string",
            "moeda": "string",
        })
        grupo.to_parquet(
                parquet_path, engine='pyarrow', index=False,
                use_dictionary={"ano": False, "mes": False, "moeda": False},
                coerce_timestamps="ms",
                allow_truncated_timestamps=True,
            )

        qtd = len(grupo)
        total_parquet += qtd
        anos_processados.append((ano, qtd))
        print(f" Ano ref: {ano}: {qtd:,} registros → {parquet_path}")



Lendo Json
Gerando arquivos Parquet por ano:
 Ano ref: 1985: 252 registros → silver-json-to_parquet/cotacao_parquet/ano=1985/dados_cotacao_1985.parquet
 Ano ref: 1986: 250 registros → silver-json-to_parquet/cotacao_parquet/ano=1986/dados_cotacao_1986.parquet
 Ano ref: 1987: 248 registros → silver-json-to_parquet/cotacao_parquet/ano=1987/dados_cotacao_1987.parquet
 Ano ref: 1988: 248 registros → silver-json-to_parquet/cotacao_parquet/ano=1988/dados_cotacao_1988.parquet
 Ano ref: 1989: 248 registros → silver-json-to_parquet/cotacao_parquet/ano=1989/dados_cotacao_1989.parquet
 Ano ref: 1990: 245 registros → silver-json-to_parquet/cotacao_parquet/ano=1990/dados_cotacao_1990.parquet
 Ano ref: 1991: 251 registros → silver-json-to_parquet/cotacao_parquet/ano=1991/dados_cotacao_1991.parquet
 Ano ref: 1992: 250 registros → silver-json-to_parquet/cotacao_parquet/ano=1992/dados_cotacao_1992.parquet
 Ano ref: 1993: 250 registros → silver-json-to_parquet/cotacao_parquet/ano=1993/dados_cotacao_1993.

<H2> Validando diferencas entre o que foi lindo e o que foi escrito:

In [79]:
import pandas as pd

df_json = pd.read_json("dados_cotacao.json")
df_parquet = pd.DataFrame()

inicio_ano = 1985
fim = 2025
while inicio_ano <= fim:
    ano = str(inicio_ano)
    print(f"Lendo {ano} ano")
    df_parquet_temp = pd.read_parquet(f"silver-json-to_parquet/cotacao_parquet/ano={ano}/dados_cotacao_{ano}.parquet")
    df_parquet = pd.concat([df_parquet, df_parquet_temp ], ignore_index=True)
    inicio_ano = inicio_ano + 1



Lendo 1985 ano
Lendo 1986 ano
Lendo 1987 ano
Lendo 1988 ano
Lendo 1989 ano
Lendo 1990 ano
Lendo 1991 ano
Lendo 1992 ano
Lendo 1993 ano
Lendo 1994 ano
Lendo 1995 ano
Lendo 1996 ano
Lendo 1997 ano
Lendo 1998 ano
Lendo 1999 ano
Lendo 2000 ano
Lendo 2001 ano
Lendo 2002 ano
Lendo 2003 ano
Lendo 2004 ano
Lendo 2005 ano
Lendo 2006 ano
Lendo 2007 ano
Lendo 2008 ano
Lendo 2009 ano
Lendo 2010 ano
Lendo 2011 ano
Lendo 2012 ano
Lendo 2013 ano
Lendo 2014 ano
Lendo 2015 ano
Lendo 2016 ano
Lendo 2017 ano
Lendo 2018 ano
Lendo 2019 ano
Lendo 2020 ano
Lendo 2021 ano
Lendo 2022 ano
Lendo 2023 ano
Lendo 2024 ano
Lendo 2025 ano


In [80]:
print("JSON:", len(df_json))
print("Parquet:", len(df_parquet))
print("Consistente:", len(df_json) == len(df_parquet))

JSON: 10228
Parquet: 10228
Consistente: True


In [81]:
df_parquet

,moeda,data_cotacao,valor,ano,mes
0,USD,1985-12-31,"3,7964E-09",1985,12
1,USD,1985-12-30,"3,7964E-09",1985,12
2,USD,1985-12-27,"3,7727E-09",1985,12
3,USD,1985-12-26,"3,7491E-09",1985,12
4,USD,1985-12-24,"3,7236E-09",1985,12
...,...,...,...,...,...
10223,USD,2025-01-08,"6,1315",2025,1
10224,USD,2025-01-07,"6,0735",2025,1
10225,USD,2025-01-06,"6,1113",2025,1
10226,USD,2025-01-03,"6,1557",2025,1


In [82]:
df_json

,moeda,data_cotacao,valor
0,USD,24/10/2025,"5,3791"
1,USD,23/10/2025,"5,3834"
2,USD,22/10/2025,"5,3892"
3,USD,21/10/2025,"5,3842"
4,USD,20/10/2025,"5,3765"
...,...,...,...
10223,USD,08/01/1985,"1,1738E-09"
10224,USD,07/01/1985,"1,1738E-09"
10225,USD,04/01/1985,"1,1520E-09"
10226,USD,03/01/1985,"1,1520E-09"
